# Run data analysis on the filtered & imputed HFNO tables

In [ ]:
# Imports:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import shap

from datetime import timezone

UTC = timezone.utc

# from lifelines import KaplanMeierFitter, CoxPHFitter
%pip install lifelines
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test

from datetime import datetime, timedelta
# from psmpy import PsmPy
# from psmpy.plotting import *

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate, train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn import preprocessing, metrics
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier

import xgboost as xgb
from xgboost import XGBClassifier

from experiment_config import *
from utils import *
%matplotlib inline

In [ ]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', None)

## 1. Read table

In [ ]:
# tuples_temp = pd.read_csv('original_episode_tuples.csv')

# dataset = "mimiciv"
# data_path = f"./filtered_data/{dataset}_tabledata_imputed_v2.csv"

# df_orig = pd.read_csv(data_path)
# df = df_orig.merge(tuples_temp, on=['final_starttime', 'final_endtime'], how='inner')

# df

In [ ]:
dataset = "mimiciv"
data_path = f"./filtered_data/{dataset}_tabledata_imputed.csv"

df = pd.read_csv(data_path)
df

In [ ]:
len(df)

In [ ]:
# df['hfno_failure'] = np.where(
#     (df['intubated'] == 1) | (df['died'] == 1), 
#     1,
#     0
# )
# df

In [ ]:
time_data = df[['final_starttime', 'final_endtime', 'deathtime', 'iv_time', 'intubated', 'died', 'hfno_failure', 'sepsis3']]
# time_data = df[['final_starttime', 'final_endtime', 'deathtime', 'iv_time', 'intubated', 'died', 'hfno_failure', 'max_flow_rate', 'sepsis3']]
# time_data = df[['final_starttime', 'final_endtime', 'deathtime', 'iv_time', 'intubated', 'died', 'hfno_failure', 'sepsis3']]
data = df.iloc[:, 5:]
# data = data.drop(columns=['hfno_failure', 'died', 'max_flow_rate', 'inhosp_mortality'])

# When using hfno_failure:
data = data.drop(columns=['intubated', 'died', 'inhosp_mortality'])

data

In [ ]:
time_data

## 1.1 Set experimental parameters

In [ ]:
TEST_SPLIT_FRAC = 0.15
# TEST_SPLIT_FRAC=0.2
MAX_SHAP_DISPLAY = 12
RUN_XGBOOST_CV = False

N_ITER_BOOTSTRAP = 10
# N_ITER_BOOTSTRAP = 100

In [ ]:
# Label translation dictionary:
label_dict = {
        # Demographics
        'gender': 'Male',
        'admission_age': 'Admission age',
        'weight_admit': 'Weight',
        'race_cat': 'Ethnicity',
        'race_cat_ASIAN': 'Asian (ethnicity)',
        'race_cat_BLACK': 'Black (ethnicity)',
        'race_cat_HISPANIC': 'Hispanic (ethnicity)',
        'race_cat_OTHER': 'Other (ethnicity)',
        'race_cat_UNKNOWN': 'Unknown (ethnicity)',
        'race_cat_WHITE': 'White (ethnicity)',
        'gcs_binned_Severe (3-8)': 'Severe GCS score (3-8)',
        'gcs_binned_Moderate (9-12)': 'Moderate GCS score (9-12)',
        'gcs_binned_Mild (13-14)': 'Mild GCS score (13-14)',
        'gcs_binned_Normal (15)': 'Normal GCS score (15)',

        # Vasopressor Usage
        'vp_last6h': 'Documented vasopressor use (last 6h)',

        # Comorbidities
        'sepsis3': 'Sepsis',
        'myocardial_infarct': 'Myocardial infarct',
        'congestive_heart_failure': 'Congestive heart failure',
        'peripheral_vascular_disease': 'Peripheral vascular disease',
        'cerebrovascular_disease': 'Cerebrovascular disease',
        'chronic_pulmonary_disease': 'COPD',
        'liver_disease': 'Liver disease',
        'renal_disease': 'Renal disease',
        'malignant_cancer': 'Malignant cancer',
        'diabetes': 'Diabetes',
        
        # Scores
        'gcs_binned': 'Glascow Coma Score',
        'apsiii': 'APS-III score',

        # Vital signs
        'heart_rate_mean_last24h': 'Mean heart rate',
        'mbp_mean_last24h': 'Mean MBP',
        'glucose_mean_last24h': 'Mean glucose',
        'platelet_last': 'Mean platelet',
        'spo2_mean_last24h': 'Mean SpO2',
        'resp_rate_mean_last24h': 'Mean respiratory rate',
        'temperature_mean_last24h': 'Mean temperature',
        'urineoutput_24hr': 'Urine output (ml)',
        'fluidbalance_24hr': 'Fluid balance (ml)',
        
        # Lab values and blood gasses
        'bicarbonate_last': 'Bicarbonate',
        'po2_last': 'pO2',
        'pco2_last': 'pCO2',
        'spo2_last': 'SpO2',
        'ptt_last': 'PTT',
        'inr_last': 'INR',
        'calcium_last': 'Calcium',
        'potassium_last': 'Potassium',
        'mchc_last': 'MCHC',
        'mch_last': 'MCH',
        'ph_last': 'pH',
        'aniongap_last': 'Anion Gap',
        'sodium_last': 'Sodium',
        'hemoglobin_last': 'Hemoglobin',
        'wbc_last': 'WBC',
        'rdw_last': 'RDW',
        'creatinine_last': 'Creatinine',

        # Settings
        'flow_rate_last': 'Flow-rate',
        'fio2_last': 'FiO2',

        'rox_last': 'ROX',

        # Secondary outcomes
        'inhosp_mortality': 'In-hospital mortality',
    }

## 1.2 Split data into training and test dataset

In [ ]:
data

In [ ]:
data.hfno_failure.sum()

In [ ]:
data.drop(columns=['flow_rate_last', 'fio2_last', 'spo2_last'], inplace=True)

In [ ]:
X = data.iloc[:,1:].to_numpy()
y = data.iloc[:,0].to_numpy()
feature_names = data.columns[1:].tolist()

for i, col in enumerate(feature_names):
    if col in label_dict.keys():
        feature_names[i] = label_dict[col]

# The outcome variable 'hfno_failure' is located at the last column.
# X = data.iloc[:,:-1].to_numpy()
# y = data.iloc[:,-1].to_numpy()
# feature_names = data.columns[:-1].tolist()

print(X)
print(y)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SPLIT_FRAC, random_state=42)
print(f"Fraction positive labels in training dataset: {np.mean(y_train)}")
print(f"Fraction positive labels in test dataset: {np.mean(y_test)}")

In [ ]:
X_train

## 2. Train predictors and visualize SHAP values

### 2.1 Perform Logistic Regression

In [ ]:
# Scale the data before fitting model:
scaler = preprocessing.StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_train_scaled

X_test_scaled = scaler.transform(X_test)

scaler_fullset = preprocessing.StandardScaler().fit(X)
X_scaled = scaler_fullset.transform(X)
X_scaled

In [ ]:
# Using training and test splits:
lr_model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000000
)

lr_model, lr_preds, _ = mc_bootstrap(X_train, y_train, X_test, y_test, lr_model, model_label='LR', n_iter=N_ITER_BOOTSTRAP)

# lr_params = {
#     'class_weight': 'balanced',
#     'max_iter': 1000000,
#     'solver': 'liblinear'
# }

# clf2, clf2_preds, clf2_results = fit_and_eval(X_train, y_train, X_test, y_test, LogisticRegression, lr_params, model_label='LR', n_iter=N_ITER)

In [ ]:
shap.initjs()
explainer = shap.Explainer(lr_model, X_train, feature_names=feature_names)
shap_values = explainer(X_test)
# shap.plots.beeswarm(shap_values)

In [ ]:
shap.summary_plot(shap_values, X_test, max_display=MAX_SHAP_DISPLAY)

In [ ]:
print(f"Intercept: {lr_model.intercept_}")
pd.DataFrame(zip(data.columns[1:].tolist(), lr_model.coef_[0]))
# clf.coef_

### 2.2 Apply XGBoost

In [ ]:
balance_ratio = (1 - np.mean(y_train)) / np.mean(y_train)
print(balance_ratio)

"""
{'learning_rate': 0.01,
 'max_delta_step': 0,
 'max_depth': 4,
 'min_child_weight': 8,
 'n_estimators': 100,
 'reg_alpha': 0,
 'reg_lambda': 1,
 'subsample': 0.5}
"""
# xgb_model = xgb.XGBClassifier(
#     objective='binary:logistic',
#     random_state=42,
#     scale_pos_weight=balance_ratio,
#     max_depth=4,
#     # gamma=2,
#     eta=0.01,
#     reg_alpha=0,
#     reg_lambda=1,
#     min_child_weight=8,
#     n_estimators=100,
#     learning_rate=0.05,
#     subsample=0.5,
# )


# xgb_model = xgb.XGBClassifier(
#     objective='binary:logistic',
#     random_state=42,
#     scale_pos_weight=balance_ratio,
#     # scale_pos_weight=1,
#     max_depth=8,
#     # eta=0.01,
#     reg_alpha=1,
#     reg_lambda=2,
#     min_child_weight=8,
#     n_estimators=200,
#     learning_rate=0.01,
#     subsample=0.5,
#     # max_delta_step=0.01
# )

xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    random_state=42,
    scale_pos_weight=balance_ratio,
    # scale_pos_weight=1,
    max_depth=4,
    # eta=0.01,
    reg_alpha=2,
    reg_lambda=4,
    min_child_weight=15,
    n_estimators=100,
    learning_rate=0.01,
    subsample=0.5,
    # max_delta_step=0.01
)

# xgb_model = xgb.XGBClassifier(
#     objective='binary:logistic',
#     random_state=42,
#     scale_pos_weight=balance_ratio,
#     # scale_pos_weight=1,
#     max_depth=8,
#     # eta=0.01,
#     reg_alpha=1,
#     reg_lambda=2,
#     min_child_weight=8,
#     n_estimators=200,
#     learning_rate=0.01,
#     subsample=0.5,
#     # max_delta_step=0.01
# )

eval_set = [(X_train, y_train), (X_test, y_test)]

# xgb_model.fit(
#     X_train,
#     y_train,
#     eval_set=eval_set,
#     verbose=True
# )

# xgb_preds = xgb_model.predict(X_test)
# print(xgb_preds)
# print(y_test)
# print()
# evaluate(y_test, xgb_preds, 'XGBoost')

xgb_model, xgb_preds, _ = mc_bootstrap(X_train, y_train, X_test, y_test, xgb_model, model_label='XGBoost', eval_set=eval_set, verbose=True, n_iter=N_ITER_BOOTSTRAP)
# xgb_model, xgb_preds, _ = mc_bootstrap(X_train, y_train, X_train, y_train, xgb_model, model_label='XGBoost', eval_set=eval_set, verbose=True, n_iter=N_ITER_BOOTSTRAP)
# fit_and_eval(X_train, y_train, X_test, y_test, xgb.XGBClassifier, xgb_params, model_label='XGBoost', eval_set=eval_set, verbose=True, n_iter=15)

In [ ]:
xgb_explainer = shap.Explainer(xgb_model, X_train, feature_names=feature_names)
xgb_shap_values = xgb_explainer(X_test, check_additivity=False)

In [ ]:
shap.summary_plot(xgb_shap_values, X_test, max_display=MAX_SHAP_DISPLAY, show=False)
plt.savefig('results/XGBoost_SHAP.pdf', bbox_inches='tight')
plt.savefig('results/XGBoost_SHAP.png', bbox_inches='tight')

In [ ]:
if RUN_XGBOOST_CV:
    param_grid = {
        # 'max_depth':[4,6,8,10],
        'max_depth':[4,8,10],
        'min_child_weight':[6,8],
        'n_estimators':[100,150,200],
        'learning_rate':[0.01, 0.05],
        'reg_alpha':[0, 1],
        'reg_lambda':[1, 2],
        'subsample':[0.5, 0.8],
        'max_delta_step':[0, 0.05]
    }
    estimator = xgb.XGBClassifier(
        objective= 'binary:logistic',
        # nthread=4,
        scale_pos_weight=balance_ratio,
        seed=42
    )
    gsearch = GridSearchCV(
        estimator = estimator, 
        param_grid = param_grid,
        scoring='roc_auc',
        n_jobs=4,
        # iid=False,
        cv=4,
        verbose=True
    )
    xgb_model = gsearch.fit(
        X_train,
        y_train,
        verbose=True
    )
    
    xgb_preds = xgb_model.predict(X_test)
    print(xgb_preds)
    print(y_test)
    print(xgb_model.best_params_)


In [ ]:
if RUN_XGBOOST_CV:
    xgb_cm = metrics.confusion_matrix(y_test, xgb_preds)
    xgb_score = xgb_model.score(X_test, y_test)
    # xgb_auroc_score = metrics.roc_auc_score(y_test, xgb_preds)
    xgb_proba = xgb_model.predict_proba(X_test)[:, 1]
    xgb_auroc_score = metrics.roc_auc_score(y_test, xgb_proba)
    # xgb_auroc_score = metrics.roc_auc_score(xgb_preds, y_test)
    
    # Source: https://towardsdatascience.com/logistic-regression-using-python-sklearn-numpy-mnist-handwriting-recognition-matplotlib-a6b31e2b166a
    plt.figure(figsize=(6,6))
    sns.heatmap(xgb_cm, annot=True, fmt=".3f", linewidths=.5, square = True, cmap = 'Blues_r');
    plt.ylabel('Actual label');
    plt.xlabel('Predicted label');
    all_sample_title = 'Acc: {0}, AUROC: {1}'.format(round(xgb_score, 4), round(xgb_auroc_score, 4))
    plt.title(all_sample_title, size = 15);

### 2.3 Apply Random Forest

In [ ]:
rf_model = RandomForestClassifier(
    random_state=42,
    n_estimators=100,
    criterion='entropy',
    max_depth=10,
    class_weight='balanced',
    min_samples_leaf=4,
    min_samples_split=2,
    verbose=True
)

# fit_and_eval(X_train, y_train, X_test, y_test, rf_model, model_label='RF')
rf_model, rf_preds, _ = mc_bootstrap(X_train, y_train, X_test, y_test, rf_model, model_label='RF', n_iter=N_ITER_BOOTSTRAP)

In [ ]:
rf_explainer = shap.Explainer(rf_model, feature_names=feature_names)
rf_shap_values = rf_explainer.shap_values(X_test)

# shap.summary_plot(rf_shap_values, X_test)
# shap.summary_plot(rf_shap_values[1], X_test, feature_names=feature_names, max_display=MAX_SHAP_DISPLAY)

### 2.4 Apply simple decision tree

In [ ]:
dt_model = DecisionTreeClassifier(
    random_state=42,
    criterion='entropy',
    class_weight='balanced',
    # min_samples_leaf=4,
    # min_samples_split=2,
)

# fit_and_eval(X_train, y_train, X_test, y_test, dt_model, model_label='DT')
dt_model, dt_preds, _ = mc_bootstrap(X_train, y_train, X_test, y_test, dt_model, model_label='DT', n_iter=N_ITER_BOOTSTRAP)

In [ ]:
dt_explainer = shap.Explainer(dt_model, feature_names=feature_names)
dt_shap_values = dt_explainer.shap_values(X_test)

# shap.summary_plot(rf_shap_values, X_test)
# shap.summary_plot(dt_shap_values[1], X_test, feature_names=feature_names, max_display=MAX_SHAP_DISPLAY)

In [ ]:
shap.initjs()

In [ ]:
# row = -3
# shap.initjs()
# # shap.force_plot(rf_explainer.expected_value[1], rf_shap_values[1][0], X_test[0, :])
# print(rf_explainer.expected_value[1])
# rf_plot = shap.force_plot(rf_explainer.expected_value[1], rf_shap_values[1][row], X_test[row, :], matplotlib = True, show = False, feature_names=feature_names)
# rf_plot

In [ ]:
row = 0 

In [ ]:
# xgb_plot = shap.force_plot(xgb_explainer.expected_value, xgb_shap_values[row], matplotlib = True, show = False, feature_names=feature_names)
xgb_plot = shap.force_plot(xgb_shap_values[row], matplotlib = True, show = False, feature_names=feature_names)
xgb_plot
# xgb_shap_values[row]

In [ ]:
# Logistic Regression:
# lr_plot = shap.force_plot(explainer.expected_value[1]
lr_plot = shap.force_plot(shap_values[row], matplotlib = True, show = False, feature_names=feature_names)
lr_plot

In [ ]:
# dt_plot = shap.force_plot(dt_explainer.expected_value[1], dt_shap_values[1][row], X_test[row, :], matplotlib = True, show = False, feature_names=feature_names)
# dt_plot

## 3. Plot survival curves for Sepsis and non-Sepsis patients

In [ ]:
time_data['timestamp_int'] = np.where(
    time_data['intubated'] == 1, 
    (pd.to_datetime(time_data['iv_time']).sub(pd.to_datetime(time_data['final_starttime']))).dt.seconds / 60 / 60,
    (pd.to_datetime(time_data['final_endtime']).sub(pd.to_datetime(time_data['final_starttime']))).dt.seconds / 60 / 60
)

# time_data
time_data['timestamp_fail'] = np.where(
    time_data['hfno_failure'] == 1,
    np.where(
        time_data['intubated'] == 1,
        (pd.to_datetime(time_data['iv_time']).sub(pd.to_datetime(time_data['final_starttime']))).dt.seconds / 60 / 60,
        (pd.to_datetime(time_data['deathtime']).sub(pd.to_datetime(time_data['final_starttime']))).dt.seconds / 60 / 60,
    ),
    (pd.to_datetime(time_data['final_endtime']).sub(pd.to_datetime(time_data['final_starttime']))).dt.seconds / 60 / 60
)

time_data

In [ ]:
time_data.head(200)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111)

timestamp_col = 'timestamp_fail'
event_col = 'hfno_failure'
T = time_data[timestamp_col]
E = time_data[event_col]

kmf = KaplanMeierFitter()

# Define SHAP-like colors
color_blue = '#1E88E5'
color_pink = '#E91E63'

sepsis_cond = (time_data['sepsis3'] == 1)

# Plot for non-sepsis group (blue)
kmf.fit(
    durations=T[~sepsis_cond],
    event_observed=E[~sepsis_cond],
    label="Other etiologies"
)
kmf.plot_survival_function(ax=ax, color=color_blue)

# Plot for sepsis group (pink)
kmf.fit(
    durations=T[sepsis_cond],
    event_observed=E[sepsis_cond],
    label="Sepsis (Sepsis-3 definition)"
)
kmf.plot_survival_function(ax=ax, color=color_pink)

# Add hazard ratio and p-value annotation (centered)
textstr = 'HR = 2.64\np < 0.005'
props = dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.85, edgecolor='gray')
ax.text(
    0.70, 0.32, textstr,
    transform=ax.transAxes,
    fontsize=12,
    fontweight='medium',
    va='center', ha='center',
    bbox=props
)

# Improve axis labels and tick readability
ax.set_xlabel('Time elapsed since HFNO initiation (hours)', fontsize=13, labelpad=8)
ax.set_ylabel('ETI-free survival (%)', fontsize=13, labelpad=8)
ax.tick_params(axis='both', labelsize=11)

# Clean legend with better alignment
leg = ax.legend(
    title='Etiology',
    loc='lower left',
    frameon=True,
    fontsize=11,
    title_fontsize=12,
    handlelength=2.8
)
# Adjust spacing between title and entries
plt.setp(leg.get_title(), multialignment='center')

# Minimal grid
ax.grid(True, which='major', linestyle='--', alpha=0.35)

# Layout and save
plt.tight_layout()
plt.savefig('results/Sepsis3_KM_survival.pdf', bbox_inches='tight')
plt.show()


### 3.1 Apply Log Rank Test

In [ ]:
T_sep = T[sepsis_cond]
E_sep = E[sepsis_cond]

T_nsep = T[~sepsis_cond]
E_nsep = E[~sepsis_cond]

results = logrank_test(T_sep, T_nsep, event_observed_A=E_sep, event_observed_B=E_nsep)
results.print_summary()

### 3.2 Apply Cox Proportional Hazards

In [ ]:
temp = data.iloc[:, :]
temp[timestamp_col] = time_data[timestamp_col].values
temp = temp.drop(columns=['race_cat_WHITE', 'gcs_binned_Normal (15)'])
temp

In [ ]:
cph = CoxPHFitter()
# cph.fit(temp, duration_col='timestamp_int', event_col='intubated')
cph.fit(temp, duration_col=timestamp_col, event_col=event_col)
cph.print_summary()

In [ ]:
plt.subplots(figsize=(10,16))
cph.plot()

## 4. Peri-HFNO Analysis (Pre-initiation + Post-initiation windows)

Same modelling workflow as Section 2 (LR -> SHAP -> coeff table -> XGBoost -> SHAP -> RF -> SHAP -> DT -> SHAP), applied to three combined feature sets:

| Config | Features |
---|---|
| Pre + 0-4 h  | All pre-HFNO features + vitals/labs in first 4 h after HFNO start |
| Pre + 0-12 h | All pre-HFNO features + vitals/labs in first 12 h |
| Pre + 0-24 h | All pre-HFNO features + vitals/labs in first 24 h |

**Requirements:** run notebook 2_HFNO_data_processing.ipynb section 9 first to produce iltered_data/mimiciv_tabledata_post_hfno_imputed.csv.

In [ ]:
# Load processed post-HFNO data (produced by notebook 2, section 9)
df_post = pd.read_csv('./filtered_data/mimiciv_tabledata_post_hfno_imputed.csv')
post_feat_cols = [c for c in df_post.columns if c not in ('stay_id', 'final_starttime')]
print(f"Post-HFNO imputed: {df_post.shape}  |  {len(post_feat_cols)} feature columns")

# Align by row position Ã¢â‚¬â€ same cohort, same order as pre-HFNO imputed CSV
df_merged = df_post[post_feat_cols].reset_index(drop=True)

# Pre-HFNO feature matrix: reuse data exactly as built in Section 2
pre_data = data.copy().reset_index(drop=True)
print(f"Pre rows: {len(pre_data)}, Post rows: {len(df_merged)}, Match: {len(pre_data) == len(df_merged)}")

In [ ]:
def get_peri_label(col, h):
    suffix = f'_post{h}h'
    base = col.replace(suffix, '')
    mapping = {
        'heart_rate_mean':      f'HR mean [{h}h]',
        'mbp_mean':             f'MBP mean [{h}h]',
        'resp_rate_mean':       f'Resp rate mean [{h}h]',
        'temperature_mean':     f'Temperature mean [{h}h]',
        'spo2_mean':            f'SpO2 mean [{h}h]',
        'glucose_mean':         f'Glucose mean [{h}h]',
        'fluidbalance':         f'Fluid balance [{h}h]',
        'lactate_last':         f'Lactate [{h}h]',
        'ph_last':              f'pH [{h}h]',
        'so2_last':             f'SO2 [{h}h]',
        'po2_last':             f'pO2 [{h}h]',
        'pco2_last':            f'pCO2 [{h}h]',
        'baseexcess_last':      f'Base excess [{h}h]',
        'totalco2_last':        f'Total CO2 [{h}h]',
        'wbc_last':             f'WBC [{h}h]',
        'hemoglobin_last':      f'Hemoglobin [{h}h]',
        'platelet_last':        f'Platelet [{h}h]',
        'mch_last':             f'MCH [{h}h]',
        'mchc_last':            f'MCHC [{h}h]',
        'rdw_last':             f'RDW [{h}h]',
        'aniongap_last':        f'Anion gap [{h}h]',
        'bicarbonate_last':     f'Bicarbonate [{h}h]',
        'bun_last':             f'BUN [{h}h]',
        'calcium_last':         f'Calcium [{h}h]',
        'creatinine_last':      f'Creatinine [{h}h]',
        'sodium_last':          f'Sodium [{h}h]',
        'potassium_last':       f'Potassium [{h}h]',
        'inr_last':             f'INR [{h}h]',
        'ptt_last':             f'PTT [{h}h]',
        'ast_last':             f'AST [{h}h]',
        'alt_last':             f'ALT [{h}h]',
        'alp_last':             f'ALP [{h}h]',
        'ld_ldh_last':          f'LD/LDH [{h}h]',
        'bilirubin_total_last': f'Bilirubin [{h}h]',
        'gcs_min':              f'GCS min [{h}h]',
        'max_flow_rate':        f'Flow rate [{h}h]',
    }
    return mapping.get(base, col)

def get_post_cols(h):
    return [c for c in df_merged.columns
            if f'post{h}h' in c and (h == 24 or 'post24h' not in c)]

# Base columns: demographics, comorbidities, severity scores, vasopressors, GCS
# (no 24h pre-labs/vitals)
BASE_COLS = [c for c in pre_data.columns[1:] if not any([
    c.endswith('_last24h'),
    c.endswith('_last'),
    c.endswith('_first'),
    c in ('rox_last', 'fluidbalance_24hr'),
])]
print(f"Base cols ({len(BASE_COLS)}): {BASE_COLS}")

y_sc = pre_data.iloc[:, 0].values

# Ã¢â€â‚¬Ã¢â€â‚¬ Scenario 1: Pre only Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
X_s1 = pre_data.iloc[:, 1:].values
fn_s1 = [label_dict.get(c, c) for c in pre_data.columns[1:]]
X_tr_s1, X_te_s1, y_tr_s1, y_te_s1 = train_test_split(X_s1, y_sc, test_size=TEST_SPLIT_FRAC, random_state=42)
print(f"S1 Pre only Ã¢â‚¬â€ train: {len(y_tr_s1)}, test: {len(y_te_s1)}, features: {len(fn_s1)}")

# Ã¢â€â‚¬Ã¢â€â‚¬ Scenario 2: Base + 4h post Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
pc_4h = get_post_cols(4)
X_s2 = np.hstack([pre_data[BASE_COLS].values, df_merged[pc_4h].values])
fn_s2 = [label_dict.get(c, c) for c in BASE_COLS] + [get_peri_label(c, 4) for c in pc_4h]
X_tr_s2, X_te_s2, y_tr_s2, y_te_s2 = train_test_split(X_s2, y_sc, test_size=TEST_SPLIT_FRAC, random_state=42)
print(f"S2 Base+4h Ã¢â‚¬â€ train: {len(y_tr_s2)}, test: {len(y_te_s2)}, features: {len(fn_s2)}")

# Ã¢â€â‚¬Ã¢â€â‚¬ Scenario 3: Full pre + 4h post Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
X_s3 = np.hstack([pre_data.iloc[:, 1:].values, df_merged[pc_4h].values])
fn_s3 = [label_dict.get(c, c) for c in pre_data.columns[1:]] + [get_peri_label(c, 4) for c in pc_4h]
X_tr_s3, X_te_s3, y_tr_s3, y_te_s3 = train_test_split(X_s3, y_sc, test_size=TEST_SPLIT_FRAC, random_state=42)
print(f"S3 Full pre+4h Ã¢â‚¬â€ train: {len(y_tr_s3)}, test: {len(y_te_s3)}, features: {len(fn_s3)}")

# Ã¢â€â‚¬Ã¢â€â‚¬ Scenario 4: Full pre + 12h post Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
pc_12h = get_post_cols(12)
X_s4 = np.hstack([pre_data.iloc[:, 1:].values, df_merged[pc_12h].values])
fn_s4 = [label_dict.get(c, c) for c in pre_data.columns[1:]] + [get_peri_label(c, 12) for c in pc_12h]
X_tr_s4, X_te_s4, y_tr_s4, y_te_s4 = train_test_split(X_s4, y_sc, test_size=TEST_SPLIT_FRAC, random_state=42)
print(f"S4 Full pre+12h Ã¢â‚¬â€ train: {len(y_tr_s4)}, test: {len(y_te_s4)}, features: {len(fn_s4)}")

# Ã¢â€â‚¬Ã¢â€â‚¬ Scenario 5: Full pre + 24h post Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
pc_24h = get_post_cols(24)
X_s5 = np.hstack([pre_data.iloc[:, 1:].values, df_merged[pc_24h].values])
fn_s5 = [label_dict.get(c, c) for c in pre_data.columns[1:]] + [get_peri_label(c, 24) for c in pc_24h]
X_tr_s5, X_te_s5, y_tr_s5, y_te_s5 = train_test_split(X_s5, y_sc, test_size=TEST_SPLIT_FRAC, random_state=42)
print(f"S5 Full pre+24h Ã¢â‚¬â€ train: {len(y_tr_s5)}, test: {len(y_te_s5)}, features: {len(fn_s5)}")

### 4.1 1. Pre only

In [ ]:
lr_s1 = LogisticRegression(class_weight='balanced', max_iter=1000000, solver='saga')
lr_s1, lr_preds_s1, boot_lr_s1 = mc_bootstrap(X_tr_s1, y_tr_s1, X_te_s1, y_te_s1, lr_s1, model_label='LR', n_iter=N_ITER_BOOTSTRAP)

In [ ]:
plt.close('all')
explainer_s1 = shap.Explainer(lr_s1, X_tr_s1, feature_names=fn_s1)
shap_values_s1 = explainer_s1(X_te_s1)
plt.figure(figsize=(10, 5))
shap.summary_plot(shap_values_s1, X_te_s1, max_display=10, show=False)
plt.title('1. Pre only Ã¢â‚¬â€ Logistic Regression')
plt.tight_layout()
plt.show()
pd.DataFrame(zip(fn_s1, lr_s1.coef_[0]), columns=['Feature', 'Coefficient']).sort_values('Coefficient', key=abs, ascending=False)

In [ ]:
bal_s1 = (1 - np.mean(y_tr_s1)) / np.mean(y_tr_s1)
xgb_s1 = xgb.XGBClassifier(
    objective='binary:logistic', random_state=42,
    scale_pos_weight=bal_s1, max_depth=4,
    reg_alpha=2, reg_lambda=4, min_child_weight=15,
    n_estimators=100, learning_rate=0.01, subsample=0.5,
)
eval_set_s1 = [(X_tr_s1, y_tr_s1), (X_te_s1, y_te_s1)]
xgb_s1, xgb_preds_s1, boot_xgb_s1 = mc_bootstrap(X_tr_s1, y_tr_s1, X_te_s1, y_te_s1, xgb_s1, model_label='XGBoost', eval_set=eval_set_s1, verbose=True, n_iter=N_ITER_BOOTSTRAP)

In [ ]:
plt.close('all')
xgb_explainer_s1 = shap.Explainer(xgb_s1, X_tr_s1, feature_names=fn_s1)
xgb_shap_s1 = xgb_explainer_s1(X_te_s1, check_additivity=False)
plt.figure(figsize=(10, 5))
shap.summary_plot(xgb_shap_s1, X_te_s1, max_display=10, show=False)
plt.title('1. Pre only Ã¢â‚¬â€ XGBoost')
plt.tight_layout()
plt.show()

In [ ]:
rf_s1 = RandomForestClassifier(
    random_state=42, n_estimators=100, criterion='entropy',
    max_depth=10, class_weight='balanced',
    min_samples_leaf=4, min_samples_split=2, verbose=True,
)
rf_s1, rf_preds_s1, boot_rf_s1 = mc_bootstrap(X_tr_s1, y_tr_s1, X_te_s1, y_te_s1, rf_s1, model_label='RF', n_iter=N_ITER_BOOTSTRAP)

In [ ]:
plt.close('all')
rf_explainer_s1 = shap.Explainer(rf_s1, feature_names=fn_s1)
rf_shap_s1 = rf_explainer_s1.shap_values(X_te_s1)

# Handle both old (list) and new (array) SHAP output formats
rf_shap_plot = rf_shap_s1[1] if isinstance(rf_shap_s1, list) else rf_shap_s1[:, :, 1]

plt.figure(figsize=(10, 5))
shap.summary_plot(rf_shap_plot, X_te_s1, feature_names=fn_s1, max_display=10, show=False)
plt.title('1. Pre only Ã¢â‚¬â€ Random Forest')
plt.tight_layout()
plt.show()

In [ ]:
dt_s1 = DecisionTreeClassifier(random_state=42, criterion='entropy', class_weight='balanced')
dt_s1, dt_preds_s1, boot_dt_s1 = mc_bootstrap(X_tr_s1, y_tr_s1, X_te_s1, y_te_s1, dt_s1, model_label='DT', n_iter=N_ITER_BOOTSTRAP)

In [ ]:
plt.close('all')
dt_explainer_s1 = shap.Explainer(dt_s1, feature_names=fn_s1)
dt_shap_s1 = dt_explainer_s1.shap_values(X_te_s1)
dt_shap_s1_plot = dt_shap_s1[1] if isinstance(dt_shap_s1, list) else dt_shap_s1[:, :, 1]
plt.figure(figsize=(10, 5))
shap.summary_plot(dt_shap_s1_plot, X_te_s1, feature_names=fn_s1, max_display=10, show=False)
plt.title('1. Pre only Ã¢â‚¬â€ Decision Tree')
plt.tight_layout()
plt.show()

In [ ]:
aurocs_s1 = {
    'LR':      metrics.roc_auc_score(y_te_s1, lr_s1.predict_proba(X_te_s1)[:, 1]),
    'XGBoost': metrics.roc_auc_score(y_te_s1, xgb_s1.predict_proba(X_te_s1)[:, 1]),
    'RF':      metrics.roc_auc_score(y_te_s1, rf_s1.predict_proba(X_te_s1)[:, 1]),
    'DT':      metrics.roc_auc_score(y_te_s1, dt_s1.predict_proba(X_te_s1)[:, 1]),
}
print("AUROCs 1. Pre only:", aurocs_s1)

### 4.2 2. Base + 4h post

In [ ]:
lr_s2 = LogisticRegression(class_weight='balanced', max_iter=1000000, solver='saga')
lr_s2, lr_preds_s2, boot_lr_s2 = mc_bootstrap(X_tr_s2, y_tr_s2, X_te_s2, y_te_s2, lr_s2, model_label='LR', n_iter=N_ITER_BOOTSTRAP)

In [ ]:
plt.close('all')
explainer_s2 = shap.Explainer(lr_s2, X_tr_s2, feature_names=fn_s2)
shap_values_s2 = explainer_s2(X_te_s2)
plt.figure(figsize=(10, 5))
shap.summary_plot(shap_values_s2, X_te_s2, max_display=10, show=False)
plt.title('2. Base + 4h post Ã¢â‚¬â€ Logistic Regression')
plt.tight_layout()
plt.show()
pd.DataFrame(zip(fn_s2, lr_s2.coef_[0]), columns=['Feature', 'Coefficient']).sort_values('Coefficient', key=abs, ascending=False)

In [ ]:
bal_s2 = (1 - np.mean(y_tr_s2)) / np.mean(y_tr_s2)
xgb_s2 = xgb.XGBClassifier(
    objective='binary:logistic', random_state=42,
    scale_pos_weight=bal_s2, max_depth=4,
    reg_alpha=2, reg_lambda=4, min_child_weight=15,
    n_estimators=100, learning_rate=0.01, subsample=0.5,
)
eval_set_s2 = [(X_tr_s2, y_tr_s2), (X_te_s2, y_te_s2)]
xgb_s2, xgb_preds_s2, boot_xgb_s2 = mc_bootstrap(X_tr_s2, y_tr_s2, X_te_s2, y_te_s2, xgb_s2, model_label='XGBoost', eval_set=eval_set_s2, verbose=True, n_iter=N_ITER_BOOTSTRAP)

In [ ]:
plt.close('all')
xgb_explainer_s2 = shap.Explainer(xgb_s2, X_tr_s2, feature_names=fn_s2)
xgb_shap_s2 = xgb_explainer_s2(X_te_s2, check_additivity=False)
plt.figure(figsize=(10, 5))
shap.summary_plot(xgb_shap_s2, X_te_s2, max_display=10, show=False)
plt.title('2. Base + 4h post Ã¢â‚¬â€ XGBoost')
plt.tight_layout()
plt.show()

In [ ]:
rf_s2 = RandomForestClassifier(
    random_state=42, n_estimators=100, criterion='entropy',
    max_depth=10, class_weight='balanced',
    min_samples_leaf=4, min_samples_split=2, verbose=True,
)
rf_s2, rf_preds_s2, boot_rf_s2 = mc_bootstrap(X_tr_s2, y_tr_s2, X_te_s2, y_te_s2, rf_s2, model_label='RF', n_iter=N_ITER_BOOTSTRAP)

In [ ]:
plt.close('all')
rf_explainer_s2 = shap.Explainer(rf_s2, feature_names=fn_s2)
rf_shap_s2 = rf_explainer_s2.shap_values(X_te_s2)
rf_shap_s2_plot = rf_shap_s2[:, :, 1] if rf_shap_s2.ndim == 3 else rf_shap_s2[1]
plt.figure(figsize=(10, 5))
shap.summary_plot(rf_shap_s2_plot, X_te_s2, feature_names=fn_s2, max_display=10, show=False)
plt.title('2. Base + 4h post Ã¢â‚¬â€ Random Forest')
plt.tight_layout()
plt.show()

In [ ]:
dt_s2 = DecisionTreeClassifier(random_state=42, criterion='entropy', class_weight='balanced')
dt_s2, dt_preds_s2, boot_dt_s2 = mc_bootstrap(X_tr_s2, y_tr_s2, X_te_s2, y_te_s2, dt_s2, model_label='DT', n_iter=N_ITER_BOOTSTRAP)

In [ ]:
plt.close('all')
dt_explainer_s2 = shap.Explainer(dt_s2, feature_names=fn_s2)
dt_shap_s2 = dt_explainer_s2.shap_values(X_te_s2)
dt_shap_s2_plot = dt_shap_s2[:, :, 1] if dt_shap_s2.ndim == 3 else dt_shap_s2[1]
plt.figure(figsize=(10, 5))
shap.summary_plot(dt_shap_s2_plot, X_te_s2, feature_names=fn_s2, max_display=10, show=False)
plt.title('2. Base + 4h post Ã¢â‚¬â€ Decision Tree')
plt.tight_layout()
plt.show()

In [ ]:
aurocs_s2 = {
    'LR':      metrics.roc_auc_score(y_te_s2, lr_s2.predict_proba(X_te_s2)[:, 1]),
    'XGBoost': metrics.roc_auc_score(y_te_s2, xgb_s2.predict_proba(X_te_s2)[:, 1]),
    'RF':      metrics.roc_auc_score(y_te_s2, rf_s2.predict_proba(X_te_s2)[:, 1]),
    'DT':      metrics.roc_auc_score(y_te_s2, dt_s2.predict_proba(X_te_s2)[:, 1]),
}
print("AUROCs 2. Base + 4h post:", aurocs_s2)

### 4.3 3. Full pre + 4h post

In [ ]:
lr_s3 = LogisticRegression(class_weight='balanced', max_iter=1000000, solver='saga')
lr_s3, lr_preds_s3, boot_lr_s3 = mc_bootstrap(X_tr_s3, y_tr_s3, X_te_s3, y_te_s3, lr_s3, model_label='LR', n_iter=N_ITER_BOOTSTRAP)

In [ ]:
plt.close('all')
explainer_s3 = shap.Explainer(lr_s3, X_tr_s3, feature_names=fn_s3)
shap_values_s3 = explainer_s3(X_te_s3)
plt.figure(figsize=(10, 5))
shap.summary_plot(shap_values_s3, X_te_s3, max_display=10, show=False)
plt.title('3. Full pre + 4h post Ã¢â‚¬â€ Logistic Regression')
plt.tight_layout()
plt.show()
pd.DataFrame(zip(fn_s3, lr_s3.coef_[0]), columns=['Feature', 'Coefficient']).sort_values('Coefficient', key=abs, ascending=False)

In [ ]:
bal_s3 = (1 - np.mean(y_tr_s3)) / np.mean(y_tr_s3)
xgb_s3 = xgb.XGBClassifier(
    objective='binary:logistic', random_state=42,
    scale_pos_weight=bal_s3, max_depth=4,
    reg_alpha=2, reg_lambda=4, min_child_weight=15,
    n_estimators=100, learning_rate=0.01, subsample=0.5,
)
eval_set_s3 = [(X_tr_s3, y_tr_s3), (X_te_s3, y_te_s3)]
xgb_s3, xgb_preds_s3, boot_xgb_s3 = mc_bootstrap(X_tr_s3, y_tr_s3, X_te_s3, y_te_s3, xgb_s3, model_label='XGBoost', eval_set=eval_set_s3, verbose=True, n_iter=N_ITER_BOOTSTRAP)

In [ ]:
plt.close('all')
xgb_explainer_s3 = shap.Explainer(xgb_s3, X_tr_s3, feature_names=fn_s3)
xgb_shap_s3 = xgb_explainer_s3(X_te_s3, check_additivity=False)
plt.figure(figsize=(10, 5))
shap.summary_plot(xgb_shap_s3, X_te_s3, max_display=10, show=False)
plt.title('3. Full pre + 4h post Ã¢â‚¬â€ XGBoost')
plt.tight_layout()
plt.show()

In [ ]:
rf_s3 = RandomForestClassifier(
    random_state=42, n_estimators=100, criterion='entropy',
    max_depth=10, class_weight='balanced',
    min_samples_leaf=4, min_samples_split=2, verbose=True,
)
rf_s3, rf_preds_s3, boot_rf_s3 = mc_bootstrap(X_tr_s3, y_tr_s3, X_te_s3, y_te_s3, rf_s3, model_label='RF', n_iter=N_ITER_BOOTSTRAP)

In [ ]:
plt.close('all')
rf_explainer_s3 = shap.Explainer(rf_s3, feature_names=fn_s3)
rf_shap_s3 = rf_explainer_s3.shap_values(X_te_s3)
rf_shap_s3_plot = rf_shap_s3[:, :, 1] if rf_shap_s3.ndim == 3 else rf_shap_s3[1]
plt.figure(figsize=(10, 5))
shap.summary_plot(rf_shap_s3_plot, X_te_s3, feature_names=fn_s3, max_display=10, show=False)
plt.title('3. Full pre + 4h post Ã¢â‚¬â€ Random Forest')
plt.tight_layout()
plt.show()

In [ ]:
dt_s3 = DecisionTreeClassifier(random_state=42, criterion='entropy', class_weight='balanced')
dt_s3, dt_preds_s3, boot_dt_s3 = mc_bootstrap(X_tr_s3, y_tr_s3, X_te_s3, y_te_s3, dt_s3, model_label='DT', n_iter=N_ITER_BOOTSTRAP)

In [ ]:
plt.close('all')
dt_explainer_s3 = shap.Explainer(dt_s3, feature_names=fn_s3)
dt_shap_s3 = dt_explainer_s3.shap_values(X_te_s3)
dt_shap_s3_plot = dt_shap_s3[:, :, 1] if dt_shap_s3.ndim == 3 else dt_shap_s3[1]
plt.figure(figsize=(10, 5))
shap.summary_plot(dt_shap_s3_plot, X_te_s3, feature_names=fn_s3, max_display=10, show=False)
plt.title('3. Full pre + 4h post Ã¢â‚¬â€ Decision Tree')
plt.tight_layout()
plt.show()

In [ ]:
aurocs_s3 = {
    'LR':      metrics.roc_auc_score(y_te_s3, lr_s3.predict_proba(X_te_s3)[:, 1]),
    'XGBoost': metrics.roc_auc_score(y_te_s3, xgb_s3.predict_proba(X_te_s3)[:, 1]),
    'RF':      metrics.roc_auc_score(y_te_s3, rf_s3.predict_proba(X_te_s3)[:, 1]),
    'DT':      metrics.roc_auc_score(y_te_s3, dt_s3.predict_proba(X_te_s3)[:, 1]),
}
print("AUROCs 3. Full pre + 4h post:", aurocs_s3)

### 4.4 4. Full pre + 12h post

In [ ]:
lr_s4 = LogisticRegression(class_weight='balanced', max_iter=1000000, solver='saga')
lr_s4, lr_preds_s4, boot_lr_s4 = mc_bootstrap(X_tr_s4, y_tr_s4, X_te_s4, y_te_s4, lr_s4, model_label='LR', n_iter=N_ITER_BOOTSTRAP)

In [ ]:
plt.close('all')
explainer_s4 = shap.Explainer(lr_s4, X_tr_s4, feature_names=fn_s4)
shap_values_s4 = explainer_s4(X_te_s4)
plt.figure(figsize=(10, 5))
shap.summary_plot(shap_values_s4, X_te_s4, max_display=10, show=False)
plt.title('4. Full pre + 12h post Ã¢â‚¬â€ Logistic Regression')
plt.tight_layout()
plt.show()
pd.DataFrame(zip(fn_s4, lr_s4.coef_[0]), columns=['Feature', 'Coefficient']).sort_values('Coefficient', key=abs, ascending=False)

In [ ]:
bal_s4 = (1 - np.mean(y_tr_s4)) / np.mean(y_tr_s4)
xgb_s4 = xgb.XGBClassifier(
    objective='binary:logistic', random_state=42,
    scale_pos_weight=bal_s4, max_depth=4,
    reg_alpha=2, reg_lambda=4, min_child_weight=15,
    n_estimators=100, learning_rate=0.01, subsample=0.5,
)
eval_set_s4 = [(X_tr_s4, y_tr_s4), (X_te_s4, y_te_s4)]
xgb_s4, xgb_preds_s4, boot_xgb_s4 = mc_bootstrap(X_tr_s4, y_tr_s4, X_te_s4, y_te_s4, xgb_s4, model_label='XGBoost', eval_set=eval_set_s4, verbose=True, n_iter=N_ITER_BOOTSTRAP)

In [ ]:
plt.close('all')
xgb_explainer_s4 = shap.Explainer(xgb_s4, X_tr_s4, feature_names=fn_s4)
xgb_shap_s4 = xgb_explainer_s4(X_te_s4, check_additivity=False)
plt.figure(figsize=(10, 5))
shap.summary_plot(xgb_shap_s4, X_te_s4, max_display=10, show=False)
plt.title('4. Full pre + 12h post Ã¢â‚¬â€ XGBoost')
plt.tight_layout()
plt.show()

In [ ]:
rf_s4 = RandomForestClassifier(
    random_state=42, n_estimators=100, criterion='entropy',
    max_depth=10, class_weight='balanced',
    min_samples_leaf=4, min_samples_split=2, verbose=True,
)
rf_s4, rf_preds_s4, boot_rf_s4 = mc_bootstrap(X_tr_s4, y_tr_s4, X_te_s4, y_te_s4, rf_s4, model_label='RF', n_iter=N_ITER_BOOTSTRAP)

In [ ]:
plt.close('all')
rf_explainer_s4 = shap.Explainer(rf_s4, feature_names=fn_s4)
rf_shap_s4 = rf_explainer_s4.shap_values(X_te_s4)
rf_shap_s4_plot = rf_shap_s4[:, :, 1] if rf_shap_s4.ndim == 3 else rf_shap_s4[1]
plt.figure(figsize=(10, 5))
shap.summary_plot(rf_shap_s4_plot, X_te_s4, feature_names=fn_s4, max_display=10, show=False)
plt.title('4. Full pre + 12h post Ã¢â‚¬â€ Random Forest')
plt.tight_layout()
plt.show()

In [ ]:
dt_s4 = DecisionTreeClassifier(random_state=42, criterion='entropy', class_weight='balanced')
dt_s4, dt_preds_s4, boot_dt_s4 = mc_bootstrap(X_tr_s4, y_tr_s4, X_te_s4, y_te_s4, dt_s4, model_label='DT', n_iter=N_ITER_BOOTSTRAP)

In [ ]:
plt.close('all')
dt_explainer_s4 = shap.Explainer(dt_s4, feature_names=fn_s4)
dt_shap_s4 = dt_explainer_s4.shap_values(X_te_s4)
dt_shap_s4_plot = dt_shap_s4[:, :, 1] if dt_shap_s4.ndim == 3 else dt_shap_s4[1]
plt.figure(figsize=(10, 5))
shap.summary_plot(dt_shap_s4_plot, X_te_s4, feature_names=fn_s4, max_display=10, show=False)
plt.title('4. Full pre + 12h post Ã¢â‚¬â€ Decision Tree')
plt.tight_layout()
plt.show()

In [ ]:
aurocs_s4 = {
    'LR':      metrics.roc_auc_score(y_te_s4, lr_s4.predict_proba(X_te_s4)[:, 1]),
    'XGBoost': metrics.roc_auc_score(y_te_s4, xgb_s4.predict_proba(X_te_s4)[:, 1]),
    'RF':      metrics.roc_auc_score(y_te_s4, rf_s4.predict_proba(X_te_s4)[:, 1]),
    'DT':      metrics.roc_auc_score(y_te_s4, dt_s4.predict_proba(X_te_s4)[:, 1]),
}
print("AUROCs 4. Full pre + 12h post:", aurocs_s4)

### 4.5 5. Full pre + 24h post

In [ ]:
lr_s5 = LogisticRegression(class_weight='balanced', max_iter=1000000, solver='saga')
lr_s5, lr_preds_s5, boot_lr_s5 = mc_bootstrap(X_tr_s5, y_tr_s5, X_te_s5, y_te_s5, lr_s5, model_label='LR', n_iter=N_ITER_BOOTSTRAP)

In [ ]:
plt.close('all')
explainer_s5 = shap.Explainer(lr_s5, X_tr_s5, feature_names=fn_s5)
shap_values_s5 = explainer_s5(X_te_s5)
plt.figure(figsize=(10, 5))
shap.summary_plot(shap_values_s5, X_te_s5, max_display=10, show=False)
plt.title('5. Full pre + 24h post Ã¢â‚¬â€ Logistic Regression')
plt.tight_layout()
plt.show()
pd.DataFrame(zip(fn_s5, lr_s5.coef_[0]), columns=['Feature', 'Coefficient']).sort_values('Coefficient', key=abs, ascending=False)

In [ ]:
bal_s5 = (1 - np.mean(y_tr_s5)) / np.mean(y_tr_s5)
xgb_s5 = xgb.XGBClassifier(
    objective='binary:logistic', random_state=42,
    scale_pos_weight=bal_s5, max_depth=4,
    reg_alpha=2, reg_lambda=4, min_child_weight=15,
    n_estimators=100, learning_rate=0.01, subsample=0.5,
)
eval_set_s5 = [(X_tr_s5, y_tr_s5), (X_te_s5, y_te_s5)]
xgb_s5, xgb_preds_s5, boot_xgb_s5 = mc_bootstrap(X_tr_s5, y_tr_s5, X_te_s5, y_te_s5, xgb_s5, model_label='XGBoost', eval_set=eval_set_s5, verbose=True, n_iter=N_ITER_BOOTSTRAP)

In [ ]:
plt.close('all')
xgb_explainer_s5 = shap.Explainer(xgb_s5, X_tr_s5, feature_names=fn_s5)
xgb_shap_s5 = xgb_explainer_s5(X_te_s5, check_additivity=False)
plt.figure(figsize=(10, 5))
shap.summary_plot(xgb_shap_s5, X_te_s5, max_display=10, show=False)
plt.title('5. Full pre + 24h post Ã¢â‚¬â€ XGBoost')
plt.tight_layout()
plt.show()

In [ ]:
rf_s5 = RandomForestClassifier(
    random_state=42, n_estimators=100, criterion='entropy',
    max_depth=10, class_weight='balanced',
    min_samples_leaf=4, min_samples_split=2, verbose=True,
)
rf_s5, rf_preds_s5, boot_rf_s5 = mc_bootstrap(X_tr_s5, y_tr_s5, X_te_s5, y_te_s5, rf_s5, model_label='RF', n_iter=N_ITER_BOOTSTRAP)

In [ ]:
plt.close('all')
rf_explainer_s5 = shap.Explainer(rf_s5, feature_names=fn_s5)
rf_shap_s5 = rf_explainer_s5.shap_values(X_te_s5)
rf_shap_s5_plot = rf_shap_s5[:, :, 1] if rf_shap_s5.ndim == 3 else rf_shap_s5[1]
plt.figure(figsize=(10, 5))
shap.summary_plot(rf_shap_s5_plot, X_te_s5, feature_names=fn_s5, max_display=10, show=False)
plt.title('5. Full pre + 24h post Ã¢â‚¬â€ Random Forest')
plt.tight_layout()
plt.show()

In [ ]:
dt_s5 = DecisionTreeClassifier(random_state=42, criterion='entropy', class_weight='balanced')
dt_s5, dt_preds_s5, boot_dt_s5 = mc_bootstrap(X_tr_s5, y_tr_s5, X_te_s5, y_te_s5, dt_s5, model_label='DT', n_iter=N_ITER_BOOTSTRAP)

In [ ]:
plt.close('all')
dt_explainer_s5 = shap.Explainer(dt_s5, feature_names=fn_s5)
dt_shap_s5 = dt_explainer_s5.shap_values(X_te_s5)
dt_shap_s5_plot = dt_shap_s5[:, :, 1] if dt_shap_s5.ndim == 3 else dt_shap_s5[1]
plt.figure(figsize=(10, 5))
shap.summary_plot(dt_shap_s5_plot, X_te_s5, feature_names=fn_s5, max_display=10, show=False)
plt.title('5. Full pre + 24h post Ã¢â‚¬â€ Decision Tree')
plt.tight_layout()
plt.show()


In [ ]:
aurocs_s5 = {
    'LR':      metrics.roc_auc_score(y_te_s5, lr_s5.predict_proba(X_te_s5)[:, 1]),
    'XGBoost': metrics.roc_auc_score(y_te_s5, xgb_s5.predict_proba(X_te_s5)[:, 1]),
    'RF':      metrics.roc_auc_score(y_te_s5, rf_s5.predict_proba(X_te_s5)[:, 1]),
    'DT':      metrics.roc_auc_score(y_te_s5, dt_s5.predict_proba(X_te_s5)[:, 1]),
}
print("AUROCs 5. Full pre + 24h post:", aurocs_s5)

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

def fmt_ci(vals, decimals=2):
    m   = np.mean(vals)
    sd  = np.std(vals)
    lo  = m - 1.96 * sd
    hi  = m + 1.96 * sd
    return f"{m:.{decimals}f} ({lo:.{decimals}f}–{hi:.{decimals}f})"

MODEL_FULL = {
    'LR':      'Logistic regression',
    'XGBoost': 'XGBoost',
    'RF':      'Random forest',
    'DT':      'Decision tree',
}

PREDICTION_MODELS = {
    'Model 1 – Pre-HFNO only':           {'LR': boot_lr_s1, 'XGBoost': boot_xgb_s1, 'RF': boot_rf_s1, 'DT': boot_dt_s1},
    'Model 2 – Baseline + 4h on-HFNO':   {'LR': boot_lr_s2, 'XGBoost': boot_xgb_s2, 'RF': boot_rf_s2, 'DT': boot_dt_s2},
    'Model 3 – Pre-HFNO + 4h on-HFNO':   {'LR': boot_lr_s3, 'XGBoost': boot_xgb_s3, 'RF': boot_rf_s3, 'DT': boot_dt_s3},
    'Model 4 – Pre-HFNO + 12h on-HFNO':  {'LR': boot_lr_s4, 'XGBoost': boot_xgb_s4, 'RF': boot_rf_s4, 'DT': boot_dt_s4},
    'Model 5 – Pre-HFNO + 24h on-HFNO':  {'LR': boot_lr_s5, 'XGBoost': boot_xgb_s5, 'RF': boot_rf_s5, 'DT': boot_dt_s5},
}

rows = []
for model_name, classifiers in PREDICTION_MODELS.items():
    for clf_key, clf_label in MODEL_FULL.items():
        boot = classifiers[clf_key]
        rows.append({
            'Prediction model': model_name,
            'Classifier':       clf_label,
            'Sensitivity':      fmt_ci(boot['sens']),
            'Specificity':      fmt_ci(boot['spec']),
            'Accuracy':         fmt_ci(boot['acc']),
            'AUROC':            fmt_ci(boot['auroc']),
        })

table_df = pd.DataFrame(rows).set_index(['Prediction model', 'Classifier'])

print("Table 2. HFNO failure prediction model performance (bootstrap mean, 95% CI)")
print("=" * 90)
display(table_df)

table_df.to_csv('results/Table2_model_performance.csv')
print("\nSaved: results/Table2_model_performance.csv")


## 5. Combined Performance and SHAP Summary

A two-row, five-column figure:
- **Row A** — AUROC bar charts for all four classifiers across each of the five feature configurations. The best-performing model per scenario is outlined.
- **Row B** — XGBoost SHAP beeswarm plots (top 10 predictors) for each scenario.

**Variables required in memory:** urocs_s1–s5, xgb_shap_s1–s5, X_te_s1–s5, n_s1–s5.

In [ ]:
import os
import io
import string
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import shap

# ── Configuration ────────────────────────────────────────────────────────────────
N_FEATURES = 8              # reduced from 10 for readability
PANEL_DPI = 600
OUTDIR = "results"
os.makedirs(OUTDIR, exist_ok=True)

# Consistent manuscript font settings
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 8,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# Your preferred colours
color_blue = "#1E88E5"
color_pink = "#E91E63"
color_grid = "#D9D9D9"
color_axis = "#444444"

MODEL_KEYS = ["LR", "XGBoost", "RF", "DT"]
MODEL_LABELS = [
    "Logistic regression",
    "XGBoost",
    "Random forest",
    "Decision tree"
]

AUROCS_ALL = [aurocs_s1, aurocs_s2, aurocs_s3, aurocs_s4, aurocs_s5]

SHAP_DATA = [
    (xgb_shap_s1, X_te_s1, fn_s1),
    (xgb_shap_s2, X_te_s2, fn_s2),
    (xgb_shap_s3, X_te_s3, fn_s3),
    (xgb_shap_s4, X_te_s4, fn_s4),
    (xgb_shap_s5, X_te_s5, fn_s5),
]


# ── Helper: add panel label outside panel ───────────────────────────────────────
def add_panel_label(fig, ax, label, xshift=-0.012, yshift=0.008):
    bbox = ax.get_position()
    fig.text(
        bbox.x0 + xshift,
        bbox.y1 + yshift,
        label,
        ha="left",
        va="bottom",
        fontsize=11,
        fontweight="bold",
        color="black"
    )


# ── Helper: render SHAP plot as image using default SHAP colours ───────────────
def shap_to_rgba(shap_vals, X_data, feat_names, n_features, dpi):
    plt.close("all")

    shap.summary_plot(
        shap_vals,
        X_data,
        feature_names=feat_names,
        max_display=n_features,
        show=False,
        plot_size=(6.4, 5.2),   # larger internal canvas
        color_bar=True
        # no cmap argument -> default SHAP colours
    )

    fig_tmp = plt.gcf()
    fig_tmp.patch.set_facecolor("white")

    for ax_tmp in fig_tmp.get_axes():
        ax_tmp.tick_params(labelsize=8, width=0.6, length=3)
        ax_tmp.xaxis.label.set_size(8)
        ax_tmp.yaxis.label.set_size(8)

        for spine in ax_tmp.spines.values():
            spine.set_linewidth(0.5)

    buf = io.BytesIO()
    fig_tmp.savefig(
        buf,
        format="png",
        dpi=dpi,
        bbox_inches="tight",
        pad_inches=0.03,
        facecolor="white"
    )
    buf.seek(0)
    rgba = plt.imread(buf)
    plt.close(fig_tmp)

    return rgba


# ── Pre-render SHAP panels ──────────────────────────────────────────────────────
shap_imgs = [
    shap_to_rgba(sv, Xd, fn, N_FEATURES, PANEL_DPI)
    for sv, Xd, fn in SHAP_DATA
]


# ── Assemble multi-panel figure ─────────────────────────────────────────────────
N_COLS = 5
panel_letters = list(string.ascii_uppercase)

fig = plt.figure(figsize=(18.5, 10.5), facecolor="white")
gs = gridspec.GridSpec(
    2, N_COLS,
    figure=fig,
    height_ratios=[0.95, 2.2],
    hspace=0.26,
    wspace=0.42
)

# ── Panels A–E: AUROC bar charts ────────────────────────────────────────────────
for col, aurocs in enumerate(AUROCS_ALL):
    ax = fig.add_subplot(gs[0, col])

    values = [aurocs[k] for k in MODEL_KEYS]
    best_idx = int(np.argmax(values))

    bar_colors = [color_blue] * len(values)
    bar_colors[best_idx] = color_pink

    bars = ax.barh(
        MODEL_LABELS,
        values,
        color=bar_colors,
        edgecolor="black",
        linewidth=0.5,
        height=0.55,
        zorder=3
    )

    # Standardized axis 0–1
    ax.set_xlim(0, 1.0)
    ax.set_xlabel("AUROC")
    ax.set_ylabel("")
    ax.set_xticks([0.0, 0.25, 0.50, 0.75, 1.00])

    ax.tick_params(axis="both", width=0.5, length=3, color=color_axis)
    ax.grid(axis="x", color=color_grid, linewidth=0.5, zorder=0)

    # Optional reference line
    ax.axvline(0.5, color="#999999", linewidth=0.6, linestyle="--", zorder=1)

    # Value labels
    for i, (bar, val) in enumerate(zip(bars, values)):
        x_pos = min(val + 0.015, 0.97)
        ax.text(
            x_pos,
            bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}",
            va="center",
            ha="left",
            fontsize=7,
            fontweight="bold" if i == best_idx else "normal",
            color="black"
        )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(0.5)
    ax.spines["bottom"].set_linewidth(0.5)

    add_panel_label(fig, ax, panel_letters[col])

# ── Panels F–J: SHAP beeswarm plots ─────────────────────────────────────────────
for col, img in enumerate(shap_imgs):
    ax = fig.add_subplot(gs[1, col])
    ax.imshow(img, aspect="auto", interpolation="nearest")
    ax.axis("off")
    add_panel_label(fig, ax, panel_letters[col + 5])

# ── Layout tweaks ────────────────────────────────────────────────────────────────
fig.subplots_adjust(left=0.05, right=0.995, top=0.97, bottom=0.05)

# ── Save ─────────────────────────────────────────────────────────────────────────
base = os.path.join(OUTDIR, "fig2_model_performance_shap")

fig.savefig(f"{base}.pdf", bbox_inches="tight", facecolor="white")
fig.savefig(f"{base}.png", bbox_inches="tight", dpi=600, facecolor="white")
fig.savefig(f"{base}.tiff", bbox_inches="tight", dpi=600, facecolor="white")

plt.show()

print(f"Saved: {base}.pdf")
print(f"Saved: {base}.png")
print(f"Saved: {base}.tiff")

In [ ]:
import os
import io
import string
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import shap

# ── Configuration ────────────────────────────────────────────────────────────────
N_FEATURES = 8              # reduced from 10 for readability
PANEL_DPI = 600
OUTDIR = "results"
os.makedirs(OUTDIR, exist_ok=True)

# Consistent manuscript font settings
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 8,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# Your preferred colours
color_blue = "#1E88E5"
color_pink = "#E91E63"
color_grid = "#D9D9D9"
color_axis = "#444444"

MODEL_KEYS = ["LR", "XGBoost", "RF", "DT"]
MODEL_LABELS = [
    "Logistic regression",
    "XGBoost",
    "Random forest",
    "Decision tree"
]

AUROCS_ALL = [aurocs_s1, aurocs_s2, aurocs_s3, aurocs_s4, aurocs_s5]

SHAP_DATA = [
    (xgb_shap_s1, X_te_s1, fn_s1),
    (xgb_shap_s2, X_te_s2, fn_s2),
    (xgb_shap_s3, X_te_s3, fn_s3),
    (xgb_shap_s4, X_te_s4, fn_s4),
    (xgb_shap_s5, X_te_s5, fn_s5),
]


# ── Helper: add panel label outside panel ───────────────────────────────────────
def add_panel_label(fig, ax, label, xshift=-0.012, yshift=0.008):
    bbox = ax.get_position()
    fig.text(
        bbox.x0 + xshift,
        bbox.y1 + yshift,
        label,
        ha="left",
        va="bottom",
        fontsize=11,
        fontweight="bold",
        color="black"
    )


# ── Helper: render SHAP plot as image using default SHAP colours ───────────────
def shap_to_rgba(shap_vals, X_data, feat_names, n_features, dpi):
    plt.close("all")

    shap.summary_plot(
        shap_vals,
        X_data,
        feature_names=feat_names,
        max_display=n_features,
        show=False,
        plot_size=(6.4, 5.2),   # larger internal canvas
        color_bar=True
        # no cmap argument -> default SHAP colours
    )

    fig_tmp = plt.gcf()
    fig_tmp.patch.set_facecolor("white")

    for ax_tmp in fig_tmp.get_axes():
        ax_tmp.tick_params(labelsize=8, width=0.6, length=3)
        ax_tmp.xaxis.label.set_size(8)
        ax_tmp.yaxis.label.set_size(8)

        for spine in ax_tmp.spines.values():
            spine.set_linewidth(0.5)

    buf = io.BytesIO()
    fig_tmp.savefig(
        buf,
        format="png",
        dpi=dpi,
        bbox_inches="tight",
        pad_inches=0.03,
        facecolor="white"
    )
    buf.seek(0)
    rgba = plt.imread(buf)
    plt.close(fig_tmp)

    return rgba


# ── Pre-render SHAP panels ──────────────────────────────────────────────────────
shap_imgs = [
    shap_to_rgba(sv, Xd, fn, N_FEATURES, PANEL_DPI)
    for sv, Xd, fn in SHAP_DATA
]


# ── Assemble multi-panel figure ─────────────────────────────────────────────────
N_COLS = 5
panel_letters = list(string.ascii_uppercase)

fig = plt.figure(figsize=(18.5, 10.5), facecolor="white")
gs = gridspec.GridSpec(
    2, N_COLS,
    figure=fig,
    height_ratios=[0.95, 2.2],
    hspace=0.26,
    wspace=0.42
)

# ── Panels A–E: AUROC bar charts ────────────────────────────────────────────────
for col, aurocs in enumerate(AUROCS_ALL):
    ax = fig.add_subplot(gs[0, col])

    values = [aurocs[k] for k in MODEL_KEYS]
    best_idx = int(np.argmax(values))

    bar_colors = [color_blue] * len(values)
    bar_colors[best_idx] = color_pink

    bars = ax.barh(
        MODEL_LABELS,
        values,
        color=bar_colors,
        edgecolor="black",
        linewidth=0.5,
        height=0.55,
        zorder=3
    )

    # Standardized axis 0–1
    ax.set_xlim(0, 1.0)
    ax.set_xlabel("AUROC")
    ax.set_ylabel("")
    ax.set_xticks([0.0, 0.25, 0.50, 0.75, 1.00])

    ax.tick_params(axis="both", width=0.5, length=3, color=color_axis)
    ax.grid(axis="x", color=color_grid, linewidth=0.5, zorder=0)

    # Optional reference line
    ax.axvline(0.5, color="#999999", linewidth=0.6, linestyle="--", zorder=1)

    # Value labels
    for i, (bar, val) in enumerate(zip(bars, values)):
        x_pos = min(val + 0.015, 0.97)
        ax.text(
            x_pos,
            bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}",
            va="center",
            ha="left",
            fontsize=7,
            fontweight="bold" if i == best_idx else "normal",
            color="black"
        )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(0.5)
    ax.spines["bottom"].set_linewidth(0.5)

    add_panel_label(fig, ax, panel_letters[col])

# ── Panels F–J: SHAP beeswarm plots ─────────────────────────────────────────────
for col, img in enumerate(shap_imgs):
    ax = fig.add_subplot(gs[1, col])
    ax.imshow(img, aspect="auto", interpolation="nearest")
    ax.axis("off")
    add_panel_label(fig, ax, panel_letters[col + 5])

# ── Layout tweaks ────────────────────────────────────────────────────────────────
fig.subplots_adjust(left=0.05, right=0.995, top=0.97, bottom=0.05)

# ── Save ─────────────────────────────────────────────────────────────────────────
base = os.path.join(OUTDIR, "fig2_model_performance_shap")

fig.savefig(f"{base}.pdf", bbox_inches="tight", facecolor="white")
fig.savefig(f"{base}.png", bbox_inches="tight", dpi=600, facecolor="white")
fig.savefig(f"{base}.tiff", bbox_inches="tight", dpi=600, facecolor="white")

plt.show()

print(f"Saved: {base}.pdf")
print(f"Saved: {base}.png")
print(f"Saved: {base}.tiff")

In [ ]:
import os
import string
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import shap

OUTDIR = "results"
os.makedirs(OUTDIR, exist_ok=True)

N_FEATURES = 5
PANEL_DPI  = 600

plt.rcParams.update({
    "font.family":     "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size":        12,
    "axes.labelsize":   12,
    "xtick.labelsize":  12,
    "ytick.labelsize":  12,
    "pdf.fonttype": 42,
    "ps.fonttype":  42,
})

color_blue = "#1E88E5"
color_pink = "#E91E63"
color_gray = "#888888"

MODEL_KEYS = ["LR", "XGBoost", "RF", "DT"]
MODEL_LABELS = {
    "LR":      "Logistic regression",
    "XGBoost": "XGBoost",
    "RF":      "Random forest",
    "DT":      "Decision tree",
}

BOOT_ALL = {
    1: {'LR': boot_lr_s1, 'XGBoost': boot_xgb_s1, 'RF': boot_rf_s1, 'DT': boot_dt_s1},
    2: {'LR': boot_lr_s2, 'XGBoost': boot_xgb_s2, 'RF': boot_rf_s2, 'DT': boot_dt_s2},
    3: {'LR': boot_lr_s3, 'XGBoost': boot_xgb_s3, 'RF': boot_rf_s3, 'DT': boot_dt_s3},
    4: {'LR': boot_lr_s4, 'XGBoost': boot_xgb_s4, 'RF': boot_rf_s4, 'DT': boot_dt_s4},
    5: {'LR': boot_lr_s5, 'XGBoost': boot_xgb_s5, 'RF': boot_rf_s5, 'DT': boot_dt_s5},
}

AUROCS_ALL = {
    s: {m: np.mean(BOOT_ALL[s][m]['auroc']) for m in MODEL_KEYS}
    for s in range(1, 6)
}


# ── Helpers ──────────────────────────────────────────────────────────────────────
def get_best_model(aurocs):
    return max(MODEL_KEYS, key=lambda m: aurocs[m])

def get_existing_var(name):
    return globals()[name] if name in globals() else None

def normalize_binary_shap_values(shap_vals, X_data):
    if hasattr(shap_vals, "values"):
        shap_vals = shap_vals.values
    if isinstance(shap_vals, list):
        shap_vals = shap_vals[1] if len(shap_vals) == 2 else shap_vals[-1]
    shap_vals = np.asarray(shap_vals)
    if shap_vals.ndim == 3:
        shap_vals = shap_vals[:, :, 1] if shap_vals.shape[2] == 2 else shap_vals[:, :, -1]
    if shap_vals.ndim != 2:
        raise ValueError(f"Expected 2D SHAP values, got shape {shap_vals.shape}")
    if shap_vals.shape[1] != X_data.shape[1]:
        raise ValueError(f"SHAP/X_data feature count mismatch: {shap_vals.shape} vs {X_data.shape}")
    return shap_vals

def get_shap_tuple(scenario_id, model_key):
    suffix = f"s{scenario_id}"
    shap_name = f"{'xgb' if model_key == 'XGBoost' else model_key.lower()}_shap_{suffix}"
    X_name    = f"X_te_{suffix}"
    fn_name   = f"fn_{suffix}"
    shap_vals  = get_existing_var(shap_name)
    X_data     = get_existing_var(X_name)
    feat_names = get_existing_var(fn_name)
    if shap_vals is None or X_data is None or feat_names is None:
        raise ValueError(f"Missing SHAP variables for {model_key}, scenario {scenario_id}: "
                         f"{shap_name}, {X_name}, {fn_name}")
    return normalize_binary_shap_values(shap_vals, X_data), X_data, feat_names

def add_panel_label(ax, label):
    ax.text(-0.10, 1.05, label, transform=ax.transAxes,
            ha="left", va="bottom", fontsize=10, fontweight="bold",
            color="black", clip_on=False)


# ── Pre-compute SHAP data ────────────────────────────────────────────────────────
best_models = {}
shap_data   = {}

for scenario_id in range(1, 6):
    bm = get_best_model(AUROCS_ALL[scenario_id])
    best_models[scenario_id] = bm
    shap_data[scenario_id]   = get_shap_tuple(scenario_id, bm)

print("Best model per prediction model:")
for s in range(1, 6):
    print(f"  Model {s}: {MODEL_LABELS[best_models[s]]}")


# ── Build figure ─────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 21.0), facecolor="white")

# Use 3 columns:
# column 0 = AUROC
# column 1 = empty spacer
# column 2 = SHAP
gs = gridspec.GridSpec(
    5, 3,
    figure=fig,
    width_ratios=[1.15, 0.65, 2.35],
    hspace=0.60,
    wspace=0.05
)

letters = list(string.ascii_uppercase)
order   = ["DT", "RF", "XGBoost", "LR"]

for row, scenario_id in enumerate(range(1, 6)):
    aurocs     = AUROCS_ALL[scenario_id]
    boots      = BOOT_ALL[scenario_id]
    best_model = best_models[scenario_id]

    # ── Left: AUROC lollipop with 95% CI ────────────────────────────────────────
    ax_l = fig.add_subplot(gs[row, 0])
    y    = np.arange(len(order))[::-1]

    for i, model in enumerate(order):
        boot  = boots[model]
        auroc = np.mean(boot["auroc"])
        sd    = np.std(boot["auroc"])
        ci_lo = auroc - 1.96 * sd
        ci_hi = auroc + 1.96 * sd
        col   = color_pink if model == best_model else color_blue

        ax_l.hlines(
            y[i], xmin=0, xmax=auroc,
            color=col, linewidth=2.0, alpha=0.85, zorder=2
        )

        ax_l.errorbar(
            auroc, y[i],
            xerr=[[auroc - ci_lo], [ci_hi - auroc]],
            fmt="none",
            color=col,
            elinewidth=1.0,
            capsize=3,
            capthick=0.8,
            alpha=0.7,
            zorder=3
        )

        ax_l.plot(
            auroc, y[i],
            marker="o",
            markersize=6.5,
            color=col,
            markeredgecolor="black",
            markeredgewidth=0.4,
            zorder=4
        )

        ax_l.text(
            min(ci_hi + 0.022, 0.97),
            y[i],
            f"{auroc:.2f}",
            va="center",
            ha="left",
            fontsize=7.5,
            fontweight="bold" if model == best_model else "normal",
            color="black"
        )

    ax_l.set_yticks(y)
    ax_l.set_yticklabels([MODEL_LABELS[m] for m in order])
    ax_l.set_xlim(0, 1)
    ax_l.set_ylim(-0.6, len(order) - 0.4)
    ax_l.set_xlabel("AUROC (95% CI)")
    ax_l.set_xticks([0.00, 0.25, 0.50, 0.75, 1.00])
    ax_l.set_xticklabels(["0", "0.25", "0.50", "0.75", "1.00"])

    ax_l.axvline(
        0.5,
        color=color_gray,
        linestyle="--",
        linewidth=0.6,
        zorder=1
    )

    ax_l.grid(axis="x", color="#DDDDDD", linewidth=0.5, zorder=0)

    ax_l.spines["top"].set_visible(False)
    ax_l.spines["right"].set_visible(False)
    ax_l.spines["left"].set_linewidth(0.5)
    ax_l.spines["bottom"].set_linewidth(0.5)
    ax_l.tick_params(axis="both", width=0.5, length=2.5)

    add_panel_label(ax_l, letters[row * 2])

    # ── Middle: empty spacer column ─────────────────────────────────────────────
    ax_gap = fig.add_subplot(gs[row, 1])
    ax_gap.axis("off")

    # ── Right: SHAP drawn directly into subplot axes ────────────────────────────
    ax_r = fig.add_subplot(gs[row, 2])
    plt.sca(ax_r)

    shap_vals, X_data, feat_names = shap_data[scenario_id]

    shap.summary_plot(
        shap_vals,
        X_data,
        feature_names=feat_names,
        max_display=N_FEATURES,
        show=False,
        plot_size=None,
        color_bar=True,
    )

    # Make SHAP lettering smaller so labels do not dominate the figure
    ax_r.tick_params(axis="both", labelsize=9, width=0.5, length=2.5)
    ax_r.xaxis.label.set_size(9)
    ax_r.yaxis.label.set_size(9)

    ax_r.spines["top"].set_visible(False)
    ax_r.spines["right"].set_visible(False)
    ax_r.spines["left"].set_linewidth(0.5)
    ax_r.spines["bottom"].set_linewidth(0.5)

    add_panel_label(ax_r, letters[row * 2 + 1])


# ── Save ─────────────────────────────────────────────────────────────────────────
fig.subplots_adjust(
    left=0.06,
    right=0.985,
    top=0.985,
    bottom=0.035
)

base = os.path.join(OUTDIR, "fig2_model_performance_shap_final")

fig.savefig(f"{base}.pdf", bbox_inches="tight", facecolor="white")
fig.savefig(f"{base}.png", dpi=PANEL_DPI, bbox_inches="tight", facecolor="white")
fig.savefig(f"{base}.tiff", dpi=PANEL_DPI, bbox_inches="tight", facecolor="white")

plt.show()

print(f"Saved: {base}.pdf / .png / .tiff")


In [ ]:
import os
import string
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import shap

# ── Output/settings ─────────────────────────────────────────────────────────────
OUTDIR = "results"
os.makedirs(OUTDIR, exist_ok=True)

N_FEATURES = 5
PANEL_DPI  = 600

plt.rcParams.update({
    "font.family":     "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size":       12,
    "axes.labelsize":  12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "pdf.fonttype":    42,
    "ps.fonttype":     42,
})

color_blue = "#1E88E5"
color_pink = "#E91E63"
color_gray = "#888888"

MODEL_KEYS = ["LR", "XGBoost", "RF", "DT"]
MODEL_LABELS = {
    "LR":      "Logistic regression",
    "XGBoost": "XGBoost",
    "RF":      "Random forest",
    "DT":      "Decision tree",
}

# ── Bootstrap objects already created in your notebook ─────────────────────────
BOOT_ALL = {
    1: {'LR': boot_lr_s1, 'XGBoost': boot_xgb_s1, 'RF': boot_rf_s1, 'DT': boot_dt_s1},
    2: {'LR': boot_lr_s2, 'XGBoost': boot_xgb_s2, 'RF': boot_rf_s2, 'DT': boot_dt_s2},
    3: {'LR': boot_lr_s3, 'XGBoost': boot_xgb_s3, 'RF': boot_rf_s3, 'DT': boot_dt_s3},
    4: {'LR': boot_lr_s4, 'XGBoost': boot_xgb_s4, 'RF': boot_rf_s4, 'DT': boot_dt_s4},
    5: {'LR': boot_lr_s5, 'XGBoost': boot_xgb_s5, 'RF': boot_rf_s5, 'DT': boot_dt_s5},
}

AUROCS_ALL = {
    s: {m: np.mean(BOOT_ALL[s][m]["auroc"]) for m in MODEL_KEYS}
    for s in range(1, 6)
}


# ── Helpers ────────────────────────────────────────────────────────────────────
def get_best_model(aurocs):
    return max(MODEL_KEYS, key=lambda m: aurocs[m])

def get_existing_var(name):
    return globals()[name] if name in globals() else None

def normalize_binary_shap_values(shap_vals, X_data):
    """
    Convert SHAP output into a 2D array of shape:
    n_samples × n_features
    """
    # shap.Explanation
    if hasattr(shap_vals, "values"):
        shap_vals = shap_vals.values

    # list output, e.g. [class0, class1]
    if isinstance(shap_vals, list):
        shap_vals = shap_vals[1] if len(shap_vals) == 2 else shap_vals[-1]
        if hasattr(shap_vals, "values"):
            shap_vals = shap_vals.values

    shap_vals = np.asarray(shap_vals)

    # 3D output: n_samples × n_features × n_classes
    if shap_vals.ndim == 3:
        if shap_vals.shape[2] == 2:
            shap_vals = shap_vals[:, :, 1]
        else:
            shap_vals = shap_vals[:, :, -1]

    # alternative 3D layout: n_classes × n_samples × n_features
    if shap_vals.ndim == 3:
        if shap_vals.shape[0] == 2:
            shap_vals = shap_vals[1, :, :]
        else:
            shap_vals = shap_vals[-1, :, :]

    if shap_vals.ndim != 2:
        raise ValueError(f"Expected 2D SHAP values, got shape {shap_vals.shape}")

    if shap_vals.shape[1] != X_data.shape[1]:
        raise ValueError(
            f"SHAP/X_data feature count mismatch: {shap_vals.shape} vs {X_data.shape}"
        )

    return shap_vals

def get_shap_tuple(scenario_id, model_key):
    suffix = f"s{scenario_id}"

    model_prefix = {
        "LR": "lr",
        "XGBoost": "xgb",
        "RF": "rf",
        "DT": "dt"
    }[model_key]

    shap_name = f"{model_prefix}_shap_{suffix}"
    X_name    = f"X_te_{suffix}"
    fn_name   = f"fn_{suffix}"

    shap_vals  = get_existing_var(shap_name)
    X_data     = get_existing_var(X_name)
    feat_names = get_existing_var(fn_name)

    if shap_vals is None or X_data is None or feat_names is None:
        raise ValueError(
            f"Missing SHAP variables for {model_key}, scenario {scenario_id}: "
            f"{shap_name}, {X_name}, {fn_name}"
        )

    shap_vals = normalize_binary_shap_values(shap_vals, X_data)
    return shap_vals, X_data, feat_names

def add_panel_label(ax, label):
    ax.text(
        -0.10, 1.05, label,
        transform=ax.transAxes,
        ha="left", va="bottom",
        fontsize=10, fontweight="bold",
        color="black", clip_on=False
    )


# ── Pre-compute best models and SHAP inputs ────────────────────────────────────
best_models = {}
shap_data   = {}

for scenario_id in range(1, 6):
    bm = get_best_model(AUROCS_ALL[scenario_id])
    best_models[scenario_id] = bm
    shap_data[scenario_id] = get_shap_tuple(scenario_id, bm)

print("Best model per prediction model:")
for s in range(1, 6):
    print(f"  Model {s}: {MODEL_LABELS[best_models[s]]}")


# ── Build figure ────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(15, 21.0), facecolor="white")

# 3 columns:
# 0 = AUROC
# 1 = spacer
# 2 = SHAP
# SHAP column made a bit narrower than before
gs = gridspec.GridSpec(
    5, 3,
    figure=fig,
    width_ratios=[1.20, 0.80, 1.95],
    hspace=0.60,
    wspace=0.05
)

letters = list(string.ascii_uppercase)
order   = ["DT", "RF", "XGBoost", "LR"]

for row, scenario_id in enumerate(range(1, 6)):
    aurocs     = AUROCS_ALL[scenario_id]
    boots      = BOOT_ALL[scenario_id]
    best_model = best_models[scenario_id]

    # ── Left panel: AUROC lollipop with percentile 95% CI ─────────────────────
    ax_l = fig.add_subplot(gs[row, 0])
    y = np.arange(len(order))[::-1]

    for i, model in enumerate(order):
        boot_vals = np.asarray(boots[model]["auroc"])
        auroc = np.mean(boot_vals)
        ci_lo, ci_hi = np.percentile(boot_vals, [2.5, 97.5])
        col = color_pink if model == best_model else color_blue

        ax_l.hlines(
            y[i], xmin=0, xmax=auroc,
            color=col, linewidth=2.0, alpha=0.85, zorder=2
        )

        ax_l.errorbar(
            auroc, y[i],
            xerr=[[auroc - ci_lo], [ci_hi - auroc]],
            fmt="none",
            color=col,
            elinewidth=1.0,
            capsize=3,
            capthick=0.8,
            alpha=0.8,
            zorder=3
        )

        ax_l.plot(
            auroc, y[i],
            marker="o",
            markersize=6.5,
            color=col,
            markeredgecolor="black",
            markeredgewidth=0.4,
            zorder=4
        )

        ax_l.text(
            min(ci_hi + 0.022, 0.97),
            y[i],
            f"{auroc:.2f}",
            va="center",
            ha="left",
            fontsize=7.5,
            fontweight="bold" if model == best_model else "normal",
            color="black"
        )

    ax_l.set_yticks(y)
    ax_l.set_yticklabels([MODEL_LABELS[m] for m in order])
    ax_l.set_xlim(0, 1)
    ax_l.set_ylim(-0.6, len(order) - 0.4)
    ax_l.set_xlabel("AUROC (95% CI)")
    ax_l.set_xticks([0.00, 0.25, 0.50, 0.75, 1.00])
    ax_l.set_xticklabels(["0", "0.25", "0.50", "0.75", "1.00"])

    ax_l.axvline(0.5, color=color_gray, linestyle="--", linewidth=0.6, zorder=1)
    ax_l.grid(axis="x", color="#DDDDDD", linewidth=0.5, zorder=0)

    ax_l.spines["top"].set_visible(False)
    ax_l.spines["right"].set_visible(False)
    ax_l.spines["left"].set_linewidth(0.5)
    ax_l.spines["bottom"].set_linewidth(0.5)
    ax_l.tick_params(axis="both", width=0.5, length=2.5)

    add_panel_label(ax_l, letters[row * 2])

    # ── Spacer column ───────────────────────────────────────────────────────────
    ax_gap = fig.add_subplot(gs[row, 1])
    ax_gap.axis("off")

    # ── Right panel: SHAP plot ─────────────────────────────────────────────────
    ax_r = fig.add_subplot(gs[row, 2])
    plt.sca(ax_r)

    shap_vals, X_data, feat_names = shap_data[scenario_id]

    shap.summary_plot(
        shap_vals,
        X_data,
        feature_names=feat_names,
        max_display=N_FEATURES,
        show=False,
        plot_size=None,
        color_bar=True,
    )

    # Slightly larger SHAP text
    ax_r.tick_params(axis="both", labelsize=10, width=0.5, length=2.5)
    ax_r.xaxis.label.set_size(10)
    ax_r.yaxis.label.set_size(10)

    for lbl in ax_r.get_yticklabels():
        lbl.set_fontsize(10)
    for lbl in ax_r.get_xticklabels():
        lbl.set_fontsize(10)

    ax_r.spines["top"].set_visible(False)
    ax_r.spines["right"].set_visible(False)
    ax_r.spines["left"].set_linewidth(0.5)
    ax_r.spines["bottom"].set_linewidth(0.5)

    add_panel_label(ax_r, letters[row * 2 + 1])


# ── Final layout / save ────────────────────────────────────────────────────────
fig.subplots_adjust(
    left=0.06,
    right=0.985,
    top=0.985,
    bottom=0.035
)

base = os.path.join(OUTDIR, "fig2_model_performance_shap_final")

fig.savefig(f"{base}.pdf", bbox_inches="tight", facecolor="white")
fig.savefig(f"{base}.png", dpi=PANEL_DPI, bbox_inches="tight", facecolor="white")
fig.savefig(f"{base}.tiff", dpi=PANEL_DPI, bbox_inches="tight", facecolor="white")

plt.show()

print(f"Saved: {base}.pdf / .png / .tiff")